# Proyecto Integrador — Semana 02  
# 03 Gold Análisis de Fraude — Daniel Guzmán

## Objetivo

Construir tablas Gold orientadas al análisis de fraude a partir de la tabla Silver.

## Entrada

- `workspace.silver.transactions_daniel`

## Salidas Gold

- `workspace.gold.resumen_fraude_daniel`
- `workspace.gold.fraude_por_dimension_daniel`
- `workspace.gold.fraude_por_monto_daniel`

## Nota metodológica

Como existen transacciones sin etiqueta de fraude, las tasas de fraude se calculan usando únicamente transacciones etiquetadas (`is_fraud = 0` o `is_fraud = 1`) como denominador.

In [0]:
from pyspark.sql import functions as F

MI_NOMBRE = "daniel"
CATALOG = "workspace"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

df_silver = spark.table(f"{CATALOG}.silver.transactions_{MI_NOMBRE}")

print(f"Filas Silver: {df_silver.count():,}")
print(f"Columnas Silver: {len(df_silver.columns)}")

display(df_silver.limit(5))

In [0]:
# Gold 1 — Resumen general de fraude

df_resumen = (
    df_silver
    .agg(
        F.count("transaction_id").alias("total_transacciones"),
        F.sum(F.when(F.col("is_fraud") == 1, 1).otherwise(0)).alias("total_fraudes"),
        F.sum(F.when(F.col("is_fraud") == 0, 1).otherwise(0)).alias("total_legitimas"),
        F.sum(F.when(F.col("is_fraud").isNull(), 1).otherwise(0)).alias("sin_label"),
        F.sum(F.when(F.col("is_fraud").isin(0, 1), 1).otherwise(0)).alias("transacciones_etiquetadas"),
        F.round(F.sum(F.when(F.col("is_fraud") == 1, F.col("amount_abs")).otherwise(0)), 2).alias("monto_total_fraudulento"),
        F.round(F.sum("amount_abs"), 2).alias("monto_total")
    )
    .withColumn(
        "tasa_fraude_global_pct",
        F.round(F.col("total_fraudes") / F.col("transacciones_etiquetadas") * 100, 4)
    )
    .withColumn(
        "pct_monto_fraudulento",
        F.round(F.col("monto_total_fraudulento") / F.col("monto_total") * 100, 4)
    )
)

df_resumen.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.gold.resumen_fraude_{MI_NOMBRE}")

display(df_resumen)

In [0]:
def fraude_por_dimension(df, columna_dimension, nombre_dimension):
    return (
        df
        .groupBy(F.col(columna_dimension).cast("string").alias("valor"))
        .agg(
            F.count("transaction_id").alias("total_transacciones"),
            F.sum(F.when(F.col("is_fraud") == 1, 1).otherwise(0)).alias("total_fraudes"),
            F.sum(F.when(F.col("is_fraud") == 0, 1).otherwise(0)).alias("total_legitimas"),
            F.sum(F.when(F.col("is_fraud").isNull(), 1).otherwise(0)).alias("sin_label"),
            F.sum(F.when(F.col("is_fraud").isin(0, 1), 1).otherwise(0)).alias("transacciones_etiquetadas"),
            F.round(F.avg("amount_abs"), 2).alias("monto_promedio")
        )
        .withColumn("dimension", F.lit(nombre_dimension))
        .withColumn(
            "tasa_fraude_pct",
            F.when(
                F.col("transacciones_etiquetadas") > 0,
                F.round(F.col("total_fraudes") / F.col("transacciones_etiquetadas") * 100, 4)
            ).otherwise(None)
        )
        .select(
            "dimension",
            "valor",
            "total_transacciones",
            "total_fraudes",
            "total_legitimas",
            "sin_label",
            "transacciones_etiquetadas",
            "tasa_fraude_pct",
            "monto_promedio"
        )
    )

In [0]:
# Gold 2 — Fraude por múltiples dimensiones

df_dim_card_type = fraude_por_dimension(df_silver, "card_type", "card_type")
df_dim_mcc = fraude_por_dimension(df_silver, "merchant_category", "merchant_category")
df_dim_hora = fraude_por_dimension(df_silver, "hora", "hora_dia")
df_dim_dia = fraude_por_dimension(df_silver, "dia_semana", "dia_semana")
df_dim_weekend = fraude_por_dimension(df_silver, "es_fin_de_semana", "es_fin_de_semana")

df_fraude_dimension = (
    df_dim_card_type
    .unionByName(df_dim_mcc)
    .unionByName(df_dim_hora)
    .unionByName(df_dim_dia)
    .unionByName(df_dim_weekend)
    .orderBy("dimension", F.col("tasa_fraude_pct").desc())
)

df_fraude_dimension.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.gold.fraude_por_dimension_{MI_NOMBRE}")

display(df_fraude_dimension.limit(50))

In [0]:
# Gold 3 — Fraude por rango de monto

df_silver_buckets = df_silver.withColumn(
    "rango_monto",
    F.when(F.col("amount_abs") < 10, "< $10")
     .when(F.col("amount_abs") < 50, "$10 - $50")
     .when(F.col("amount_abs") < 100, "$50 - $100")
     .when(F.col("amount_abs") < 500, "$100 - $500")
     .when(F.col("amount_abs") < 1000, "$500 - $1000")
     .otherwise("> $1000")
)

df_gold_monto = (
    df_silver_buckets
    .groupBy("rango_monto")
    .agg(
        F.count("transaction_id").alias("total_transacciones"),
        F.sum(F.when(F.col("is_fraud") == 1, 1).otherwise(0)).alias("total_fraudes"),
        F.sum(F.when(F.col("is_fraud") == 0, 1).otherwise(0)).alias("total_legitimas"),
        F.sum(F.when(F.col("is_fraud").isNull(), 1).otherwise(0)).alias("sin_label"),
        F.sum(F.when(F.col("is_fraud").isin(0, 1), 1).otherwise(0)).alias("transacciones_etiquetadas"),
        F.round(F.avg("amount_abs"), 2).alias("monto_promedio")
    )
    .withColumn(
        "tasa_fraude_pct",
        F.when(
            F.col("transacciones_etiquetadas") > 0,
            F.round(F.col("total_fraudes") / F.col("transacciones_etiquetadas") * 100, 4)
        ).otherwise(None)
    )
    .orderBy(
        F.when(F.col("rango_monto") == "< $10", 1)
         .when(F.col("rango_monto") == "$10 - $50", 2)
         .when(F.col("rango_monto") == "$50 - $100", 3)
         .when(F.col("rango_monto") == "$100 - $500", 4)
         .when(F.col("rango_monto") == "$500 - $1000", 5)
         .otherwise(6)
    )
)

df_gold_monto.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.gold.fraude_por_monto_{MI_NOMBRE}")

display(df_gold_monto)

In [0]:
print("Tablas Gold creadas:")
display(spark.sql(f"SHOW TABLES IN {CATALOG}.gold"))

print("Resumen fraude:")
display(spark.table(f"{CATALOG}.gold.resumen_fraude_{MI_NOMBRE}"))

print("Top dimensiones con mayor tasa de fraude:")
display(
    spark.table(f"{CATALOG}.gold.fraude_por_dimension_{MI_NOMBRE}")
    .filter(F.col("transacciones_etiquetadas") >= 100)
    .orderBy(F.col("tasa_fraude_pct").desc())
    .limit(10)
)

print("Fraude por monto:")
display(spark.table(f"{CATALOG}.gold.fraude_por_monto_{MI_NOMBRE}"))

## Documentación Gold — Análisis de Fraude

Se construyeron tres tablas Gold orientadas a responder preguntas de negocio sobre fraude:

### `workspace.gold.resumen_fraude_daniel`

Resume el total de transacciones, total de fraudes, tasa global de fraude, monto total transaccionado y porcentaje del monto fraudulento.

### `workspace.gold.fraude_por_dimension_daniel`

Consolida métricas de fraude para distintas dimensiones:

- Tipo de tarjeta.
- Categoría de comercio.
- Hora del día.
- Día de la semana.
- Fin de semana vs día hábil.

Se usa una estructura unificada con las columnas `dimension` y `valor`, lo que facilita consultar varios cortes analíticos desde una sola tabla.

### `workspace.gold.fraude_por_monto_daniel`

Agrupa las transacciones por rangos de monto para analizar si el fraude se concentra en transacciones pequeñas, medianas o altas.

### Criterio para tasas

La tasa de fraude se calculó usando únicamente transacciones etiquetadas (`is_fraud = 0` o `is_fraud = 1`).  
Los registros sin etiqueta se conservaron, pero no se usaron como denominador para no diluir la tasa.

## Resultados — Resumen general de fraude

El dataset Silver contiene **13,305,915 transacciones**.

De estas, **8,914,963 transacciones tienen etiqueta de fraude**, mientras que **4,390,952 transacciones no tienen label**.  
Por esta razón, la tasa de fraude global se calculó usando únicamente las transacciones etiquetadas.

### Métricas principales

- Total de transacciones: **13,305,915**
- Transacciones etiquetadas: **8,914,963**
- Transacciones sin label: **4,390,952**
- Total de fraudes: **13,332**
- Total de legítimas: **8,901,631**
- Tasa global de fraude: **0.1495%**
- Monto total fraudulento: **1,749,004.78**
- Monto total transaccionado: **706,873,606.96**
- Porcentaje del monto fraudulento: **0.2474%**

### Interpretación

La tasa global de fraude es baja en cantidad de transacciones, aproximadamente **0.1495%** sobre las transacciones etiquetadas.

Sin embargo, el porcentaje del monto fraudulento es **0.2474%**, un poco mayor que la tasa por cantidad. Esto indica que, aunque los fraudes son pocos en volumen de registros, su impacto económico proporcional es ligeramente mayor.